# 🧫 SQANTI3 QC Classification File Worksheet

This worksheet is designed to help you explore and interpret the output of the SQANTI3 Quality Control module, specifically the `classification.txt` file. Use Python, bash, or any preferred method to answer the following questions.

---

## 🔍 **Basic Exploration**

1. **How many total transcript isoforms are present in the file?**

In [1]:
suppressMessages(library(readr))
suppressMessages(library(dplyr))

# Load the datasets
basic.df <- read_tsv("results/01_QC_basic/human_chr8_classification.txt")
complete.df <- read_tsv("results/02_QC_with_orthogonal/human_chr8_classification.txt")

nrow(basic.df)

[1] 2452



Rows: 2452 Columns: 54
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (12): isoform, chrom, strand, structural_category, subcategory, FSM_clas...
dbl (21): start, end, length, exons, ref_length, ref_exons, diff_to_TSS, dif...
lgl (21): RTS_stage, min_sample_cov, min_cov, min_cov_pos, sd_cov, FL, n_ind...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 2452 Columns: 60
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (14): isoform, chrom, strand, structural_category, subcategory, FSM_clas...
dbl (34): start, end, length, exons, ref_length, ref_exons, diff_to_TSS, dif...
lgl (12): RTS_stage, n_indels, n_indels_junc, bite, iso_exp, gene_exp, ratio...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types =

<details><summary>Answer</summary>2452 isoforms</details><br>

2. **How many unique genes are represented?**  
   *(Use the `associated_gene` column.)*

In [2]:
basic.df %>% select(associated_gene) %>%
    distinct() %>%
    nrow()

[1] 796


<details><summary>Answer</summary>796 unique reference genes</details><br>

3. **What are the different `structural_category` values present, and how many isoforms fall into each?**

In [3]:
basic.df %>%
    select(structural_category) %>%
    table()

.
              antisense       full-splice_match                  fusion 
                     51                    1333                       1 
                  genic            genic_intron incomplete-splice_match 
                     43                     115                     364 
             intergenic        novel_in_catalog    novel_not_in_catalog 
                     46                     330                     169 


<details><summary>Answer</summary>

- full-splice_match: 1333
- incomplete-splice_match: 364
- novel_in_catalog: 330
- novel_not_in_catalog: 169
- genic_intron: 115
- antisense: 51
- intergenic: 46
- genic: 43
- fusion: 1
</details><br>

---

## 📊 **Gene and Transcript Structure**

4. **What is the average number of exons per transcript?**  
   *(Use the `exons` column.)*

In [4]:
basic.df %>% pull(exons) %>% mean()

[1] 7.371126


<details><summary>Answer</summary>7.37 exons on average</details><br>

5. **Identify the transcript with the highest number of exons. What is its structural category and associated gene?**

In [5]:
basic.df %>% 
    filter(exons == max(basic.df$exons)) %>%
    select(isoform,exons,structural_category,associated_gene)

# A tibble: 1 × 4
  isoform             exons structural_category associated_gene  
  <chr>               <dbl> <chr>               <chr>            
1 ENSG00000253729.9_2    86 full-splice_match   ENSG00000253729.9


<details><summary>Answer</summary>ENSG00000253729.9_2 with 86 exons. Category: full-splice_match, Gene: ENSG00000253729.9</details><br>

6. **Find the longest and shortest transcripts based on the `length` column. What are their structural categories?**

In [6]:
basic.df %>% 
    filter(length == max(basic.df$length) | length == min(basic.df$length)) %>%
    select(isoform,length,structural_category,associated_gene)

# A tibble: 2 × 4
  isoform               length structural_category associated_gene   
  <chr>                  <dbl> <chr>               <chr>             
1 ENSG00000132549.20_27  13988 full-splice_match   ENSG00000132549.20
2 IT_novel_00628_0          81 genic_intron        novelGene_105     


<details><summary>Answer</summary>

Shortest:
- IT_novel_00628_0 (81 nt, genic_intron, gene: novelGene_105)

Longest:
- ENSG00000132549.20_27 (13988 nt, full-splice_match, gene: ENSG00000132549.20)
</details><br>

---

## 🧪 **Novelty and Annotations**

7. **From the `novel_not_in_catalog` isoforms, how many have all of their junctions canonical?**
    *(Use the `all_canonical` column.)*

In [7]:
basic.df %>%
    filter(structural_category == "novel_not_in_catalog") %>%
    select(all_canonical) %>%
    table()

.
    canonical non_canonical 
          152            17 


<details><summary>Answer</summary>There are 152 isoforms with all junctions canonical and 17 with at least one non-canonical junction.</details><br>

8. **From the isoforms classified as fusion, which one has the highest number of genes and what genes are they?**

In [8]:
basic.df %>% 
    filter(structural_category == "fusion") %>%
    rowwise() %>%
    mutate(fusion_genes= length(stringr::str_split(associated_gene,"_")[[1]])) %>%
    filter(fusion_genes == max(fusion_genes)) %>%
    select(isoform, fusion_genes, associated_gene)

# A tibble: 1 × 3
# Rowwise: 
  isoform             fusion_genes associated_gene                    
  <chr>                      <int> <chr>                              
1 ENSG00000253250.3_4            2 ENSG00000253250.3_ENSG00000289502.1


<details><summary>Answer</summary>There is 1 fusion transcript (ENSG00000253250.3_4), formed by 2 genes (ENSG00000253250.3 and ENSG00000289502.1).</details><br>

9. **Filter transcripts where `associated_transcript` is `novel`. What percentage of the total do they represent?**

In [9]:
basic.df %>%
    mutate(novel = ifelse(stringr::str_detect(associated_transcript,"novel"),TRUE,FALSE)) %>%
    select(novel) %>%
    table()/nrow(basic.df)*100

.
   FALSE     TRUE 
69.20881 30.79119 


<details><summary>Answer</summary>30.79% of the total</details><br>

---

## 🧪 **Coding and ORFs**

10. **How many transcripts are predicted to be coding (`coding` column)?**

In [10]:
basic.df %>% 
    select(coding) %>%
    table()

.
    coding non_coding 
      1722        730 


<details><summary>Answer</summary>1722 transcripts are predicted to be coding</details><br>

11. **Among coding transcripts, what is the average CDS length (`CDS_length` column)?**

In [11]:
basic.df %>%
    filter(coding == "coding") %>%
    pull(CDS_length) %>%
    mean()

[1] 1335.998


<details><summary>Answer</summary>1336 bp</details><br>

12. **Which transcript has the longest predicted CDS, and what is its structural category?**

In [12]:
basic.df %>%
    filter(coding == "coding") %>%
    filter(CDS_length == max(CDS_length)) %>%
    select(isoform,CDS_length,structural_category)

# A tibble: 1 × 3
  isoform             CDS_length structural_category
  <chr>                    <dbl> <chr>              
1 ENSG00000253729.9_2      12387 full-splice_match  


<details><summary>Answer</summary>ENSG00000253729.9_2, CDS length: 12387, Category: full-splice_match</details><br>

13. **How many transcripts are predicted to be subject to nonsense-mediated decay? What does it mean?**

In [13]:
basic.df %>%
    filter(predicted_NMD) %>%
    select(structural_category) %>%
    table()

.
           antisense    full-splice_match     novel_in_catalog 
                   1                   42                   35 
novel_not_in_catalog 
                  25 


<details><summary>Answer</summary>103.
    Nonsense-mediated decay (NMD) is a cellular mechanism that degrades mRNA transcripts containing premature stop codons, preventing the production of truncated proteins that could be harmful to the cell. SQANTI3 is able to flag transcripts like this if during the ORF prediction, a STOP codon is found before the TTS.
    <!-- TODO: Complete this with additional details on NMD and its implications for transcript analysis. -->
    </details><br>

---

## 🧠 **Advanced / Comparative**

Now, lets compare the classification file that used all of the orthogonal data, to see what extra information SQANTI3 is able to integrate.

14. **How many columns have been filled with information in the new classification? Name some of them.**

In [14]:
basic.df %>%
    select(where(~ !all(is.na(.)))) -> clean_basic.df

complete.df %>%
    select(where(~ !all(is.na(.)))) %>%
    colnames() %>%
    setdiff(clean_basic.df %>% colnames()) %>%
    length()

complete.df %>%
    select(where(~ !all(is.na(.)))) %>%
    colnames() %>%
    setdiff(clean_basic.df %>% colnames())

[1] 17
 [1] "min_sample_cov"                 "min_cov"                       
 [3] "min_cov_pos"                    "sd_cov"                        
 [5] "FL"                             "dist_to_CAGE_peak"             
 [7] "within_CAGE_peak"               "polyA_motif"                   
 [9] "polyA_dist"                     "polyA_motif_found"             
[11] "ratio_TSS"                      "FL.cDNA_PacBio-endo_1_coverage"
[13] "FL.cDNA_PacBio-endo_2_coverage" "FL.cDNA_PacBio-endo_3_coverage"
[15] "FL.cDNA_PacBio-h1_1_coverage"   "FL.cDNA_PacBio-h1_2_coverage"  
[17] "FL.cDNA_PacBio-h1_3_coverage"  


<details><summary>Answer</summary>
There are 17 new columns that have been filled with information now, such as:

- min_sample_cov
- min_cov
- min_cov_pos
- sd_cov
- FL
- dist_to_CAGE_peak
- within_CAGE_peak
- polyA_motif
- polyA_dist
- polyA_motif_found
- ratio_TSS
- FL.cDNA_PacBio-endo_1_coverage
- FL.cDNA_PacBio-endo_2_coverage
- FL.cDNA_PacBio-endo_3_coverage
- FL.cDNA_PacBio-h1_1_coverage
- FL.cDNA_PacBio-h1_2_coverage
- FL.cDNA_PacBio-h1_3_coverage
</details><br>

15. **From the FSM isoforms, how many have both support from a CAGE peak and a polyA motif? And the ISM?**

In [15]:
complete.df %>%
    filter(structural_category %in% c("full-splice_match","incomplete-splice_match") &
          within_CAGE_peak & polyA_motif_found) %>%
    select(structural_category) %>%
    table()

.
      full-splice_match incomplete-splice_match 
                    995                      66 


<details><summary>Answer</summary>

- FSM: 995 isoforms
- ISM: 66 isoforms
</details><br>

16. **What is the average minimum coverage of a junction for each structural category?**

In [16]:
complete.df %>%
    group_by(structural_category) %>%
    summarise(cov_mean = mean(min_cov,na.rm=TRUE),
              cov_sd = sd(min_cov,na.rm=TRUE))

# A tibble: 9 × 3
  structural_category     cov_mean cov_sd
  <chr>                      <dbl>  <dbl>
1 antisense                   8.82  12.7 
2 full-splice_match         125.   456.  
3 fusion                     10     NA   
4 genic                       0.8    1.79
5 genic_intron              NaN     NA   
6 incomplete-splice_match   324.   556.  
7 intergenic                 12     14.2 
8 novel_in_catalog           79.7  438.  
9 novel_not_in_catalog        4.52   8.55


<details><summary>Answer</summary>

| Structural Category        | cov_mean | cov_sd |
|---------------------------|----------|--------|
| antisense                 | 8.82     | 12.7   |
| full-splice_match         | 125.0    | 456.0  |
| fusion                    | 10.0     | NA     |
| genic                     | 0.80     | 1.79   |
| genic_intron              | NaN      | NA     |
| incomplete-splice_match   | 324.0    | 556.0  |
| intergenic                | 12.0     | 14.2   |
| novel_in_catalog          | 79.7     | 438.0  |
| novel_not_in_catalog      | 4.52     | 8.55   |

</details><br>

---

## 📁 **Integration**

17. **From the ISM isoforms that have support from a CAGE peak and a polyA motif, what are their subcategories? How would you explain this?.**

In [17]:
complete.df %>%
    filter(structural_category == "incomplete-splice_match") %>%
    select(subcategory) %>%
    table()

complete.df %>%
    filter(structural_category == "incomplete-splice_match" &
          within_CAGE_peak & polyA_motif_found) %>%
    select(subcategory) %>%
    table()

.
 3prime_fragment  5prime_fragment intron_retention        mono-exon 
              97               25               27              215 
.
 3prime_fragment  5prime_fragment intron_retention        mono-exon 
              15               18               25                8 


<details><summary>Answer</summary>

All ISMs:
- mono-exon: 215
- 3prime_fragment: 97
- intron_retention: 27
- 5prime_fragment: 25

ISMs supported by CAGE & polyA:
- intron_retention: 25
- 5prime_fragment: 18
- 3prime_fragment: 15
- mono-exon: 8

The fact that we see 3' fragments and 5' fragments with support in both their TSS and TTS suggests that these might be isoforms with alternative starts and end of transcription from what we can see in the reference annotation. However, the other ISMs that are not validated by the orthogonal data, might be more likely to be degradation products or artifacts.

</details><br>

18. **There is a hypothesis made in the SQANTI3 paper that states that the TSS ratio is higher on isoforms supported by a CAGE peak. Would you say that assumption is true based on the results you obtained?** Briefly explain why it would make sense or not. 

<!-- TODO: make this figure pretty -->

In [18]:
complete.df %>%
    mutate(TSS_ratio = ifelse(ratio_TSS > 1,TRUE,FALSE)) %>%
    filter(!is.na(ratio_TSS)) %>%
    select(TSS_ratio) %>%
    table()

library(ggplot2)
complete.df %>% 
    filter(!is.na(ratio_TSS)) %>%
    ggplot(aes(x=ratio_TSS,fill=within_CAGE_peak)) +
    geom_density(alpha=0.5) +
    scale_x_log10(expand = c(0, 0))  +
    scale_y_continuous(expand = c(0, 0)) +
    theme_bw() +
    theme(axis.text = element_text(size=16),
          axis.title = element_text(size=18),
          legend.text = element_text(size=16),
          legend.title = element_text(size=18)) +
    labs(x = "TSS ratio",
         y= "Density",
         fill = "Within a \nCAGE peak")

ggsave("results/03_QC_with_orthogonal/ratio_TSS_density.png")

.
FALSE  TRUE 
  385  2066 



Saving 7 x 7 in image


<details><summary>Answer</summary>
As we can see in the plot, there is a higher TSS ratio for the isoforms supported by a CAGE peak, which is consistent with the hypothesis. This is because CAGE peaks are indicative of real transcription start sites (and not artifacts of degradation), and isoforms with higher TSS ratios are more likely to be associated with such peaks.

<image src="results/03_QC_with_orthogonal/ratio_TSS_density.png" alt="TSS ratio plot" width="600"/>

</details><br>